# TensorBinding.jl — Many-Body Methods

Methods for correlated and interacting systems:

1. **Exciton Hamiltonians** — two-particle electron-hole systems encoded
   in a `2L`-qubit chain; LDoS and DoS via MPS-mode KPM.
2. **RPA susceptibility** — Dyson series resummed via the Wynn
   epsilon-algorithm.


In [ ]:
using LinearAlgebra
using Plots
using ITensors
using ITensorMPS
include("../src/TensorBinding.jl")
using .TensorBinding

---
## 1. Exciton Hamiltonians

An exciton is a correlated electron-hole pair.  In the quantics representation
the two-particle Hilbert space is encoded on a `2L`-qubit chain:
sites `1..L` for the electron, sites `L+1..2L` for the hole.

The exciton Hamiltonian is
$$H_{\rm exc} = H_e \otimes I_h + I_e \otimes H_h + V_{eh}$$
where $V_{eh}(r_e, r_h) = -U/|r_e - r_h|$ is the Coulomb attraction.

The `2L`-site MPO makes the standard MPO-mode Chebyshev list expensive;
the **MPS mode** (`KPM_Tn(H, Ncheb, X; ...)`) propagates a single reference
state $|X, X\rangle$ and is the recommended approach.

---
## 2. Exciton Hamiltonians

An exciton is a correlated electron-hole pair.  In the quantics representation
the two-particle Hilbert space is encoded on a `2L`-qubit chain:
sites `1..L` for the electron, sites `L+1..2L` for the hole.

The exciton Hamiltonian is
$$H_{\rm exc} = H_e \otimes I_h + I_e \otimes H_h + V_{eh}$$
where $V_{eh}(r_e, r_h) = -U/|r_e - r_h|$ is the Coulomb attraction.

The `2L`-site MPO makes the standard MPO-mode Chebyshev list expensive;
the **MPS mode** (`KPM_Tn(H, Ncheb, X; ...)`) propagates a single reference
state $|X, X\rangle$ and is the recommended approach.

In [ ]:
# 1D electron + hole on a chain; Coulomb attraction
L_exc   = 5          # 2^5 = 32 sites per particle
t_exc   = 1.0
U_exc   = 2.0        # Coulomb strength

H_exc = TensorBinding.get_Hamiltonian("exciton_1d", (t=t_exc, U=U_exc); L=L_exc)
println(H_exc)

In [ ]:
# MPS-mode KPM: propagate |X,X> (electron and hole both at site X)
X_exc   = H_exc.N ÷ 2     # exciton centre
Ncheb_e = 80
mdim_e  = 60
TensorBinding.KPM_Tn(H_exc, Ncheb_e, X_exc; maxdim=mdim_e)

In [ ]:
# Exciton LDOS at |X,X>
omega_exc = range(-6.0, 6.0; length=120)
ldos_exc  = [TensorBinding.get_ldos_online(H_exc, Ncheb_e, X_exc, [omega_p]; maxdim=mdim_e)[1]
             for omega_p in omega_exc]

plot(collect(omega_exc), ldos_exc;
     xlabel="energy", ylabel="LDOS", title="Exciton LDOS at X=$(X_exc)",
     legend=false, lw=2)

---
## 2. RPA susceptibility via Wynn ε-algorithm

The Random Phase Approximation resums the Dyson series for the interacting
susceptibility:
$$\chi = \chi_0 + \chi_0 U \chi_0 + \cdots = \frac{\chi_0}{1 - U\chi_0}$$

The bare $\chi_0(q,\omega)$ is computed from the KPM density matrix and Green's
function.  Wynn's epsilon algorithm accelerates convergence of the partial sums.

---
## 5. RPA susceptibility via Wynn ε-algorithm

The Random Phase Approximation resums the Dyson series for the interacting
susceptibility:

$$\chi_\text{RPA} = \Pi_0 + \Pi_0 V \Pi_0 + \Pi_0 V \Pi_0 V \Pi_0 + \cdots
= (I - \Pi_0 V)^{-1} \Pi_0$$

where $\Pi_0(\omega)$ is the non-interacting polarization bubble and $V$ is the
bare interaction.

**Efficient workflow** (two improvements over the raw MPO approach):

1. **Density matrix once** — $P = \theta(\mu - H)$ is computed a single time via
   McWeeny purification and cached in `H._density_cache`.  All frequencies share
   the same $P$; no KPM loop is needed.

2. **Wynn ε-algorithm** — instead of solving the large linear system $(I - \Pi_0 V)\chi = \Pi_0$
   at every $\omega$, we build the Neumann series
   $T_0 = \Pi_0$, $T_n = T_{n-1} V \Pi_0$, extract scalars
   $s_n(q) = -\operatorname{Im}\langle q | T_n | q \rangle / \pi$ via `get_spect_k`,
   and apply the Wynn ε-algorithm to the partial-sum sequence $[S_0, S_1, \ldots, S_K]$
   per $(q, \omega)$.  With $K = 6$ terms, the Padé estimate $\varepsilon_6$ typically
   converges as well as hundreds of MPO solves.

All of this is exposed through the single call `get_rpa_susceptibility_wynn`.

In [3]:
# ── System ────────────────────────────────────────────────────────────────────
# Reuse H1 from section 1 (L=5, N=32, 1D nearest-neighbour chain).
# The density matrix is computed once via McWeeny purification and
# stored in H1._density_cache.  Every call to get_bubble_mpo with
# P_method=:purification will find it there and skip recomputation.

println("Pre-computing P via McWeeny purification...")
TensorBinding.mcweeny_purify(H1; maxiters=40, maxdim=100, cutoff=1e-8, tol=1e-5, verbose=false)
println("Tr(P) = ", real(tr(H1._density_cache)),
        "  (target N/2 = ", H1.N ÷ 2, ")")
println("maxlinkdim(P) = ", ITensorMPS.maxlinkdim(H1._density_cache))

# ── Interaction ───────────────────────────────────────────────────────────────
# On-site Hubbard-U: V(i,j) = U·δ(i,j)  →  MPOV = U × identity MPO
U_hub = 2.0
MPOV  = U_hub * MPO(H1.sites, "Id")
println("\nInteraction: on-site Hubbard U = $U_hub")

Pre-computing P via McWeeny purification...


Tr(P) = 15.99996522081863  (target N/2 = 16)
maxlinkdim(P) = 13

Interaction: on-site Hubbard U = 2.0


In [ ]:
q_axis = 0:nq-1
ω_axis = collect(ωlist_rpa)

# ── Panel 1: Π₀ vs best Wynn estimate ─────────────────────────────────────────
# chi_partial[1,...] = partial sum at K=0, i.e. bare Π₀(q,ω)
# chi_wynn[n_wynn,...] = highest Padé estimate, using K_max+1 terms total
p_pi0 = heatmap(q_axis, ω_axis, chi_partial[1, :, :];
    title="Π₀(q,ω)  [bare bubble]",
    xlabel="q", ylabel="ω", color=:inferno)

p_rpa = heatmap(q_axis, ω_axis, chi_wynn[n_wynn, :, :];
    title="χ_RPA(q,ω)  [Wynn ε_$(2n_wynn), $(K_max+1) terms]",
    xlabel="q", ylabel="ω", color=:inferno)

display(plot(p_pi0, p_rpa; layout=(1, 2), size=(860, 360),
    plot_title="RPA susceptibility  (U = $U_hub, N = $(H1.N))"))

# ── Panel 2: Convergence of partial sums vs Wynn estimates ────────────────────
# Total spectral weight Σ_{q,ω} |−Im χ| at each approximation level
spec_partial = [sum(abs.(chi_partial[k+1, :, :])) for k in 0:K_max]
spec_wynn    = [sum(abs.(chi_wynn[m, :, :]))       for m in 1:n_wynn]

p_conv = plot(0:K_max, spec_partial;
    label="Partial sum K", lw=2, marker=:circle, color=:crimson,
    xlabel="Order K  (number of terms = K+1)",
    ylabel="Σ_{q,ω} |−Im χ|",
    title="Convergence: Neumann series vs Wynn ε-acceleration")

for m in 1:n_wynn
    scatter!(p_conv, [2m], [spec_wynn[m]];
        label="Wynn ε_$(2m)  ($(2m+1) terms)",
        markersize=9, markershape=:star5)
end

hline!(p_conv, [spec_wynn[end]]; lw=1, ls=:dot, color=:gray,
    label="Wynn ε_$(2n_wynn) (best estimate)")

display(plot(p_conv; size=(680, 400)))